# Feedforward Networks II: The Backpropagation Algorithm
## Chain rule, computational graphs, and gradient flow

**MAT 4953/6973 — Mathematical Foundations of AI** (Spring 2026, UTSA)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduenez/MathAIspring2026UTSA/blob/main/code/backpropagation.ipynb)

---

> **Prerequisites:** [`fwdpass_units.ipynb`](fwdpass_units.ipynb) (forward pass, activation functions), multivariate derivatives (partial derivatives, chain rule), Python/NumPy.

Training a network requires computing $\partial\mathcal{L}/\partial W^{(l)}$ and $\partial\mathcal{L}/\partial\mathbf{b}^{(l)}$ for **every** weight and bias across all layers, at every training step.
For the MNIST model there are 55,050 parameters — so we need 55,050 partial derivatives, repeated for every mini-batch.

**The naive approach (finite differences)** estimates each derivative as:
$$\frac{\partial\mathcal{L}}{\partial w_i} \approx \frac{\mathcal{L}(\mathbf{w} + \varepsilon\mathbf{e}_i) - \mathcal{L}(\mathbf{w})}{\varepsilon},$$
but requires one full forward pass per parameter. For $P = 55{,}050$ parameters that is $55{,}050$ forward passes per gradient step — completely impractical.

**Backpropagation** exploits the chain rule to compute *all* $P$ partial derivatives in a *single* backward pass — the same cost as one forward pass. This is its fundamental miracle.

**Outline**
1. [The chain rule of calculus](#part1)
2. [Computational graphs](#part2)
3. [Backward pass: upstream × local](#part3)
4. [Backpropagation in a feedforward network](#part4)
5. [NumPy implementation and gradient check](#part5)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, Circle, FancyBboxPatch
from ipywidgets import interact, IntSlider
from scipy.special import ndtr
from scipy.stats import norm as _norm

plt.rcParams.update({'font.size': 12, 'axes.titlesize': 13, 'figure.dpi': 100})

# Activation functions (same as fwdpass_units.ipynb)
def sigmoid(z):    return 1 / (1 + np.exp(-z))
def d_sigmoid(z):  s = sigmoid(z); return s * (1 - s)
def relu(z):       return np.maximum(0.0, z) * 1
def d_relu(z):     return (z > 0) * 1
def softmax(z):
    e = np.exp(z - np.max(z)); return e / e.sum()
def cross_entropy(y_hat, y_true):
    return -np.sum(y_true * np.log(y_hat + 1e-12))

print("Libraries loaded.")

<a id="part1"></a>
# Part 1: The Chain Rule of Calculus

## 1.1 Single-variable chain rule

If $a = f(z)$ and $z = g(x)$, then $a = f(g(x))$ and:

$$\boxed{\frac{da}{dx} = \frac{da}{dz}\cdot\frac{dz}{dx}.}$$

**Example:** $a = \sigma(z) = \frac{1}{1+e^{-z}}$ and $z = wx + b$:
$$\frac{da}{dx} = \underbrace{\sigma'(z)}_{\text{local gradient at }\sigma} \cdot \underbrace{w}_{\text{local gradient at }z}.$$

## 1.2 Multivariate chain rule

If $z = f(x_1, \ldots, x_n)$ and each $x_i = g_i(t)$, then:
$$\frac{\partial z}{\partial t} = \sum_{i=1}^{n} \frac{\partial z}{\partial x_i}\cdot\frac{\partial x_i}{\partial t}.$$

This is the key formula for backpropagation: a node that *receives* input from multiple upstream sources accumulates gradient contributions from all of them.

**Vector form.** If $\mathbf{a} = f(\mathbf{z})$ where $\mathbf{z} = g(\mathbf{w})$, the chain rule gives:
$$\frac{\partial \mathcal{L}}{\partial \mathbf{w}} = \underbrace{\left(\frac{\partial \mathbf{z}}{\partial \mathbf{w}}\right)^{\!\top}}_{\text{Jacobian}^{\top}} \frac{\partial \mathcal{L}}{\partial \mathbf{z}}.$$

In [ ]:
# Visualise the chain rule: L(w) = cross_entropy(softmax(w*x + b), y)
# for a single scalar weight w, with x, b, y fixed

x_fixed = 2.0
b_fixed = -0.5
y_true  = np.array([1.0, 0.0])   # true class = 0

w_range = np.linspace(-3, 3, 300)

losses = []
for w in w_range:
    z = np.array([w * x_fixed + b_fixed, 0.0])   # 2-class logits
    losses.append(cross_entropy(softmax(z), y_true))
losses = np.array(losses)

# Exact gradient at w=1.0 via chain rule
w0 = 1.0
z0  = np.array([w0 * x_fixed + b_fixed, 0.0])
yh  = softmax(z0)
# d L / d z_0  =  y_hat_0 - y_true_0  (softmax + cross-entropy gradient)
dL_dz0 = yh[0] - y_true[0]
dz0_dw = x_fixed
dL_dw  = dL_dz0 * dz0_dw

L0 = cross_entropy(yh, y_true)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(w_range, losses, color='#3498db', lw=2.5, label=r'$\mathcal{L}(w)$')
ax.scatter([w0], [L0], color='#e74c3c', s=60, zorder=5)
ax.annotate('', xy=(w0 + 0.6, L0 + 0.6 * dL_dw),
            xytext=(w0, L0),
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=2))
ax.text(w0 + 0.65, L0 + 0.6 * dL_dw + 0.04,
        fr'slope $= {dL_dw:.3f}$', color='#e74c3c', fontsize=11)
ax.set_xlabel(r'weight $w$', fontsize=13)
ax.set_ylabel(r'loss $\mathcal{L}(w)$', fontsize=13)
ax.set_title(r'Loss as a function of a single weight; red arrow = gradient direction', fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout(); plt.show()
print(f"Chain rule:  dL/dw = (dL/dz) * (dz/dw) = {dL_dz0:.4f} × {dz0_dw:.1f} = {dL_dw:.4f}")

<a id="part2"></a>
# Part 2: Computational Graphs

## 2.1 Graphs as a bookkeeping device

Any differentiable expression can be represented as a **computational graph**: a directed acyclic graph (DAG) in which
- each **leaf node** holds an input or parameter ($x$, $w$, $b$, …),
- each **interior node** applies a single elementary operation (add, multiply, $\sigma$, …), and
- **edges** carry the result of each operation forward (and, during backprop, gradients backward).

**Example.** Consider the loss for a single ReLU unit:

$$\ell = \tfrac{1}{2}(a - y)^2, \quad\text{where}\quad a = \mathrm{ReLU}(z), \quad z = wx + b.$$

The graph has the following structure (read left to right):

$$x,\, w \;\longrightarrow\; [\times] \;\longrightarrow\; [+] \;\longleftarrow\; b \quad\longrightarrow\quad [\mathrm{ReLU}] \;\longrightarrow\; [\tfrac{1}{2}(\cdot - y)^2] \;\longleftarrow\; y$$

We will use concrete values: $x = 2,\; w = 0.5,\; b = -0.3,\; y = 1.0$.

## 2.2 Forward pass: computing values

The forward pass evaluates the graph left to right, storing every intermediate value.
Each stored value will be reused during backprop.

In [ ]:
# ── Graph geometry ──────────────────────────────────────────────────────────
NODE_POS = {
    'x':    (0.0,  2.0),
    'w':    (0.0,  0.0),
    'mul':  (2.0,  1.0),
    'b':    (2.0, -1.0),
    'add':  (4.0,  1.0),
    'relu': (6.0,  1.0),
    'y':    (6.0, -1.0),
    'loss': (8.5,  1.0),
}
EDGES = [('x','mul'), ('w','mul'), ('mul','add'), ('b','add'),
         ('add','relu'), ('relu','loss'), ('y','loss')]

NODE_LABELS = {
    'x': r'$x$', 'w': r'$w$', 'b': r'$b$', 'y': r'$y$',
    'mul': r'$\times$', 'add': r'$+$',
    'relu': 'ReLU', 'loss': r'$\frac{1}{2}(a{-}y)^2$',
}
NODE_TYPE = {n: ('leaf' if n in ('x','w','b','y') else 'op') for n in NODE_POS}

# Forward-pass values
xv, wv, bv, yv = 2.0, 0.5, -0.3, 1.0
mulv = wv * xv         # = 1.0
addv = mulv + bv       # = 0.7  (this is z)
reluv = relu(addv)     # = 0.7  (this is a)
lossv = 0.5 * (reluv - yv)**2   # = 0.045

FWD_VALS = {
    'x': xv, 'w': wv, 'b': bv, 'y': yv,
    'mul': mulv, 'add': addv, 'relu': reluv, 'loss': lossv,
}
FWD_LABELS = {
    'x': f'x={xv}', 'w': f'w={wv}', 'b': f'b={bv}', 'y': f'y={yv}',
    'mul': f'wx={mulv}', 'add': f'z={addv}',
    'relu': f'a={reluv}', 'loss': f'ℓ={lossv:.3f}',
}

# Backward-pass gradients (∂ℓ/∂node)
dL_loss = 1.0
dL_relu = reluv - yv           # = -0.3   (∂ℓ/∂a)
dL_add  = dL_relu * d_relu(addv)   # = -0.3 * 1 = -0.3  (∂ℓ/∂z)
dL_mul  = dL_add * 1.0         # = -0.3   (∂ℓ/∂(wx), add has local grad 1)
dL_b    = dL_add * 1.0         # = -0.3   (∂ℓ/∂b)
dL_w    = dL_mul * xv          # = -0.6   (∂ℓ/∂w)
dL_x    = dL_mul * wv          # = -0.15  (∂ℓ/∂x)

BWD_GRADS = {
    'loss': dL_loss, 'relu': dL_relu, 'add': dL_add,
    'mul': dL_mul, 'b': dL_b, 'w': dL_w, 'x': dL_x, 'y': '—',
}

print("Forward-pass values:")
for k, v in FWD_VALS.items():
    print(f"  {NODE_LABELS[k]:>20s}  =  {v}")
print("\nBackward-pass gradients  ∂ℓ/∂(node):")
for k, v in BWD_GRADS.items():
    g = f'{v:.4f}' if isinstance(v, float) else v
    print(f"  ∂ℓ/∂({k:>5s})  =  {g}")

In [ ]:
def draw_graph(ax, reveal_fwd=None, reveal_bwd=None):
    # Draw the computational graph.
    # reveal_fwd: set of node names whose forward values are shown.
    # reveal_bwd: set of node names whose backward gradients are shown.
    ax.set_xlim(-0.8, 10.0)
    ax.set_ylim(-2.5, 3.5)
    ax.set_aspect('equal')
    ax.axis('off')

    r = 0.55   # node radius

    # Edges
    for src, dst in EDGES:
        x0, y0 = NODE_POS[src]
        x1, y1 = NODE_POS[dst]
        ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle='->', color='#555555',
                                   lw=1.4, connectionstyle='arc3,rad=0.0'))

    # Nodes
    for name, (x, y) in NODE_POS.items():
        is_leaf = NODE_TYPE[name] == 'leaf'
        if reveal_bwd and name in reveal_bwd and name != 'y':
            fc = '#fde8e8'   # red tint for backward-revealed nodes
            ec = '#c0392b'
        elif reveal_fwd and name in reveal_fwd:
            fc = '#e8f8f0'   # green tint for forward-revealed nodes
            ec = '#27ae60'
        else:
            fc = '#f5f5f5' if not is_leaf else '#eaf4ff'
            ec = '#999999'
        c = Circle((x, y), r, color=fc, ec=ec, lw=1.8, zorder=3)
        ax.add_patch(c)
        ax.text(x, y, NODE_LABELS[name], ha='center', va='center',
                fontsize=10, zorder=4)

    # Forward values (green, above nodes)
    if reveal_fwd:
        for name in reveal_fwd:
            x, y = NODE_POS[name]
            ax.text(x, y + r + 0.22, FWD_LABELS[name],
                    ha='center', va='bottom', fontsize=9,
                    color='#27ae60', fontweight='bold')

    # Backward gradients (red, below nodes)
    if reveal_bwd:
        for name in reveal_bwd:
            if name == 'y':
                continue
            x, y = NODE_POS[name]
            g = BWD_GRADS[name]
            gstr = f'$\\partial\\ell/\\partial = {g:.3f}$' if isinstance(g, float) else ''
            ax.text(x, y - r - 0.22, gstr,
                    ha='center', va='top', fontsize=8.5,
                    color='#c0392b', fontweight='bold')

# Static forward-pass view
fig, ax = plt.subplots(figsize=(11, 4))
draw_graph(ax, reveal_fwd=set(FWD_VALS.keys()))
ax.set_title('Computational graph — forward pass values', fontsize=13)
plt.tight_layout(); plt.show()

## 2.3 Interactive: step through the forward pass

Use the slider below to see values being computed left to right.
At each step, a new node is evaluated and its value stored.

In [ ]:
FWD_ORDER = ['x', 'w', 'b', 'y', 'mul', 'add', 'relu', 'loss']

def _plot_fwd(step=0):
    fig, ax = plt.subplots(figsize=(11, 4.2))
    revealed = set(FWD_ORDER[:step + 1]) if step >= 0 else set()
    draw_graph(ax, reveal_fwd=revealed)
    node_name = FWD_ORDER[min(step, len(FWD_ORDER) - 1)]
    ax.set_title(f'Forward pass  —  step {step}: computing {node_name}  '
                 f'(value = {FWD_LABELS[node_name]})', fontsize=12)
    plt.tight_layout(); plt.show()

interact(_plot_fwd, step=IntSlider(min=0, max=len(FWD_ORDER)-1,
                                    step=1, value=0, description='Step:'));

<a id="part3"></a>
# Part 3: Backward Pass — Upstream × Local

## 3.1 The backpropagation rule

At each node $v$ in the graph, backpropagation computes:

$$\boxed{\frac{\partial\mathcal{L}}{\partial v} = \underbrace{\frac{\partial\mathcal{L}}{\partial u}}_{\text{upstream gradient}} \cdot \underbrace{\frac{\partial u}{\partial v}}_{\text{local gradient}}}$$

where $u$ is the node immediately downstream of $v$. If $v$ feeds into multiple nodes $u_1, u_2, \ldots$, the contributions are **summed**:
$$\frac{\partial\mathcal{L}}{\partial v} = \sum_i \frac{\partial\mathcal{L}}{\partial u_i} \cdot \frac{\partial u_i}{\partial v}.$$

The **local gradient** $\partial u / \partial v$ depends only on the operation at node $u$ — it can be computed analytically once and reused. The **upstream gradient** $\partial\mathcal{L}/\partial u$ is whatever was computed at the downstream node in the previous backward step.

## 3.2 Local gradients for our example

| Operation | Output $u$ | Input $v$ | Local gradient $\partial u/\partial v$ |
|-----------|-----------|-----------|---------------------------------------|
| $u = w \cdot x$ | mul | $w$ | $x$ |
| $u = w \cdot x$ | mul | $x$ | $w$ |
| $u = z_{\mathrm{mul}} + b$ | add | $z_{\mathrm{mul}}$ | $1$ |
| $u = z_{\mathrm{mul}} + b$ | add | $b$ | $1$ |
| $u = \mathrm{ReLU}(z)$ | relu | $z$ | $\mathbb{1}[z > 0]$ |
| $u = \tfrac{1}{2}(a-y)^2$ | loss | $a$ | $a - y$ |

These local gradients are **the same regardless of the current parameter values** — only the values of $x$, $w$, $b$ change at each training step.

In [ ]:
BWD_ORDER = ['loss', 'relu', 'add', 'b', 'mul', 'w', 'x']

def _plot_bwd(step=0):
    fig, ax = plt.subplots(figsize=(11, 4.2))
    revealed_fwd = set(FWD_VALS.keys())           # always show all forward values
    revealed_bwd = set(BWD_ORDER[:step + 1])
    draw_graph(ax, reveal_fwd=revealed_fwd, reveal_bwd=revealed_bwd)
    node_name = BWD_ORDER[min(step, len(BWD_ORDER) - 1)]
    g = BWD_GRADS[node_name]
    gstr = f'{g:.4f}' if isinstance(g, float) else str(g)
    ax.set_title(f'Backward pass  —  step {step}: ∂ℓ/∂({node_name}) = {gstr}', fontsize=12)
    plt.tight_layout(); plt.show()

interact(_plot_bwd, step=IntSlider(min=0, max=len(BWD_ORDER)-1,
                                    step=1, value=0, description='Step:'));

> **Exercise 3.1.** *(Chain rule on the graph)*
>
> **(a)** Using the table of local gradients above, verify by hand (with $x=2, w=0.5, b=-0.3, y=1$) that:
> $$\frac{\partial\ell}{\partial w} = -0.6, \qquad \frac{\partial\ell}{\partial b} = -0.3, \qquad \frac{\partial\ell}{\partial x} = -0.15.$$
>
> **(b)** Suppose the unit uses **sigmoid** instead of ReLU. Redo the backward pass: what are $\partial\ell/\partial w$ and $\partial\ell/\partial b$ now? (Recall $\sigma'(z) = \sigma(z)(1-\sigma(z))$.)
>
> **(c)** In the backward step for the `mul` node, the local gradient with respect to $w$ is $x$ (the *value* of the other input). Explain why this means that weights receiving large-magnitude inputs tend to get large gradient updates.

<a id="part4"></a>
# Part 4: Backpropagation in a Feedforward Network

## 4.1 The $\boldsymbol{\delta}$ notation

For a network with $L$ layers we define the **error signal** at layer $l$ as:
$$\delta^{(l)} := \frac{\partial\mathcal{L}}{\partial\mathbf{z}^{(l)}} \in \mathbb{R}^{n_l}.$$

This is the gradient of the loss with respect to the *pre-activations* of layer $l$.
Using the chain rule, the $\delta$'s satisfy a backward recurrence:

**Output layer** (softmax + cross-entropy, one-hot target $\mathbf{y}$):
$$\boxed{\delta^{(L)} = \hat{\mathbf{y}} - \mathbf{y}.}$$
This elegant formula is the combined gradient of softmax and cross-entropy.

**Hidden layer** $l < L$ (element-wise activation $\sigma$):
$$\boxed{\delta^{(l)} = \left(W^{(l+1)\top}\,\delta^{(l+1)}\right) \odot \sigma'\!\left(\mathbf{z}^{(l)}\right).}$$

Here $\odot$ denotes element-wise multiplication. The two factors reflect:
- $W^{(l+1)\top}\,\delta^{(l+1)}$: upstream gradient arriving from layer $l+1$, "broadcast back" through the weight matrix (the local gradient of the linear map);
- $\sigma'(\mathbf{z}^{(l)})$: local gradient of the activation function.

## 4.2 Parameter gradients

Once the $\delta^{(l)}$ are in hand, the weight and bias gradients are:
$$\boxed{\frac{\partial\mathcal{L}}{\partial W^{(l)}} = \delta^{(l)}\!\left(\mathbf{a}^{(l-1)}\right)^{\!\top}, \qquad \frac{\partial\mathcal{L}}{\partial\mathbf{b}^{(l)}} = \delta^{(l)}.}$$

These formulas follow immediately from the chain rule applied to the affine map $\mathbf{z}^{(l)} = W^{(l)}\mathbf{a}^{(l-1)} + \mathbf{b}^{(l)}$.

**Mnemonic:** $\nabla_{W^{(l)}}\mathcal{L} = \underbrace{\delta^{(l)}}_{\text{what layer }l\text{ should have been}} \underbrace{(\mathbf{a}^{(l-1)})^{\top}}_{\text{what it received}}$.

## 4.3 Derivation of the output-layer formula

The cross-entropy loss for a one-hot target $\mathbf{y}$ is $\mathcal{L} = -\sum_k y_k \log \hat{y}_k$.
With $\hat{\mathbf{y}} = \mathrm{softmax}(\mathbf{z}^{(L)})$, a direct computation gives:

$$\frac{\partial\mathcal{L}}{\partial z_k^{(L)}} = \hat{y}_k - y_k, \qquad k = 1, \ldots, K.$$

In vector form: $\delta^{(L)} = \hat{\mathbf{y}} - \mathbf{y}$.

**Why is this so clean?** The softmax and cross-entropy gradients "cancel" each other's complexity. The Jacobian of softmax contains cross-terms $\partial\hat{y}_k/\partial z_j = -\hat{y}_k\hat{y}_j$ for $k \ne j$, but when composed with the cross-entropy gradient, all cross-terms collapse and the result is the simple residual $\hat{\mathbf{y}} - \mathbf{y}$.

> **Exercise 4.1.** *(Deriving $\delta^{(L)}$)*
>
> Derive $\delta^{(L)} = \hat{\mathbf{y}} - \mathbf{y}$ from scratch.
>
> **(a)** Write $\mathcal{L} = -\sum_k y_k \log \hat{y}_k$ and $\hat{y}_k = e^{z_k}/\sum_j e^{z_j}$.
> Compute $\partial\mathcal{L}/\partial z_k$ using the quotient rule for softmax.
>
> *Hint:* split into two cases: $j = k$ (diagonal) and $j \ne k$ (off-diagonal).
>
> **(b)** Now substitute back into $\partial\mathcal{L}/\partial z_k$ and simplify using $\sum_j y_j = 1$.

## 4.4 Derivation of the hidden-layer recurrence

For layer $l < L$, we apply the chain rule through layer $l+1$:

$$\delta_k^{(l)} = \frac{\partial\mathcal{L}}{\partial z_k^{(l)}} = \sum_{j=1}^{n_{l+1}} \frac{\partial\mathcal{L}}{\partial z_j^{(l+1)}} \cdot \frac{\partial z_j^{(l+1)}}{\partial z_k^{(l)}}.$$

Now, $z_j^{(l+1)} = \sum_k W^{(l+1)}_{jk}\,a_k^{(l)} + b_j^{(l+1)}$ and $a_k^{(l)} = \sigma(z_k^{(l)})$, so:

$$\frac{\partial z_j^{(l+1)}}{\partial z_k^{(l)}} = W^{(l+1)}_{jk}\,\sigma'\!\left(z_k^{(l)}\right).$$

Substituting:
$$\delta_k^{(l)} = \sigma'\!\left(z_k^{(l)}\right) \sum_{j} W^{(l+1)}_{jk}\,\delta_j^{(l+1)} = \sigma'\!\left(z_k^{(l)}\right)\cdot\left[W^{(l+1)\top}\delta^{(l+1)}\right]_k.$$

In vector notation: $\delta^{(l)} = \bigl(W^{(l+1)\top}\delta^{(l+1)}\bigr) \odot \sigma'(\mathbf{z}^{(l)})$.

> **Exercise 4.2.** *(Hidden-layer recurrence)*
>
> **(a)** Fill in the full derivation of $\nabla_{W^{(l)}}\mathcal{L} = \delta^{(l)}(\mathbf{a}^{(l-1)})^\top$ starting from the chain rule applied to the affine map $\mathbf{z}^{(l)} = W^{(l)}\mathbf{a}^{(l-1)} + \mathbf{b}^{(l)}$.
>
> **(b)** For ReLU activations, what is $\sigma'(\mathbf{z}^{(l)})$? What does it mean for backpropagation when many units have $z_k^{(l)} < 0$?

<a id="part5"></a>
# Part 5: NumPy Implementation and Gradient Check

We implement the full forward + backward pass on a $(3 \to 4 \to 2)$ network with ReLU hidden units and softmax+cross-entropy output — the same network traced in [`fwdpass_units.ipynb`](fwdpass_units.ipynb).

In [ ]:
np.set_printoptions(precision=6, suppress=True)
rng5 = np.random.default_rng(0)

# ── Network parameters ───────────────────────────────────────────────────────
W1 = rng5.standard_normal((4, 3)) * 0.5
b1 = np.zeros(4)
W2 = rng5.standard_normal((2, 4)) * 0.5
b2 = np.zeros(2)

# ── A single training example ────────────────────────────────────────────────
x     = np.array([0.5, -1.2, 0.8])
y_one_hot = np.array([1.0, 0.0])   # true class = 0

# ════════════════════════════════════════════════════════════════════════════
# FORWARD PASS
# ════════════════════════════════════════════════════════════════════════════
a0 = x

z1 = W1 @ a0 + b1
a1 = relu(z1)              # ReLU hidden layer

z2 = W2 @ a1 + b2
y_hat = softmax(z2)        # softmax output

loss = cross_entropy(y_hat, y_one_hot)

print("=" * 54)
print("FORWARD PASS")
print("=" * 54)
print(f"  z⁽¹⁾     = {z1}")
print(f"  a⁽¹⁾     = {a1}   ({(z1<0).sum()} units zeroed)")
print(f"  z⁽²⁾     = {z2}")
print(f"  ŷ        = {y_hat}   (sum={y_hat.sum():.6f})")
print(f"  loss     = {loss:.6f}")

# ════════════════════════════════════════════════════════════════════════════
# BACKWARD PASS
# ════════════════════════════════════════════════════════════════════════════
# Output layer: δ⁽²⁾ = ŷ - y
delta2 = y_hat - y_one_hot

# Gradients for layer 2
dW2 = np.outer(delta2, a1)     # δ⁽²⁾ (a⁽¹⁾)ᵀ
db2 = delta2

# Hidden layer: δ⁽¹⁾ = (W⁽²⁾ᵀ δ⁽²⁾) ⊙ σ'(z⁽¹⁾)
delta1 = (W2.T @ delta2) * d_relu(z1)

# Gradients for layer 1
dW1 = np.outer(delta1, a0)     # δ⁽¹⁾ (a⁽⁰⁾)ᵀ  = δ⁽¹⁾ xᵀ
db1 = delta1

print("\n" + "=" * 54)
print("BACKWARD PASS")
print("=" * 54)
print(f"  δ⁽²⁾     = {delta2}   (= ŷ - y)")
print(f"  δ⁽¹⁾     = {delta1}")
print(f"  ∇_W⁽²⁾ ℒ shape: {dW2.shape}    max|grad| = {np.abs(dW2).max():.4f}")
print(f"  ∇_W⁽¹⁾ ℒ shape: {dW1.shape}   max|grad| = {np.abs(dW1).max():.4f}")

## 5.1 Numerical gradient check

The only reliable way to verify a backprop implementation is the **gradient check**: compare the analytic gradient to a finite-difference estimate.

For parameter $\theta_i$, the two-sided finite difference approximation is:
$$\frac{\partial\mathcal{L}}{\partial \theta_i} \approx \frac{\mathcal{L}(\boldsymbol{\theta} + \varepsilon\mathbf{e}_i) - \mathcal{L}(\boldsymbol{\theta} - \varepsilon\mathbf{e}_i)}{2\varepsilon}.$$

For well-implemented backprop, the **relative error**
$$\frac{\|\nabla_{\mathrm{analytic}} - \nabla_{\mathrm{numerical}}\|}{\|\nabla_{\mathrm{analytic}}\| + \|\nabla_{\mathrm{numerical}}\|}$$
should be $\ll 10^{-5}$.

In [ ]:
def forward_loss(W1, b1, W2, b2, x, y):
    # Full forward pass, returns scalar loss.
    a1 = relu(W1 @ x + b1)
    y_hat = softmax(W2 @ a1 + b2)
    return cross_entropy(y_hat, y)

def numerical_grad(param, grad_fn, eps=1e-5):
    # Two-sided finite-difference gradient for a flat parameter vector.
    p = param.ravel().copy()
    num = np.zeros_like(p)
    for i in range(len(p)):
        p[i] += eps; lp = grad_fn(p)
        p[i] -= 2*eps; lm = grad_fn(p)
        p[i] += eps
        num[i] = (lp - lm) / (2 * eps)
    return num.reshape(param.shape)

def rel_error(a, b):
    return np.linalg.norm(a - b) / (np.linalg.norm(a) + np.linalg.norm(b) + 1e-12)

# Numerical gradients
num_dW2 = numerical_grad(W2, lambda p: forward_loss(W1, b1, p.reshape(W2.shape), b2, x, y_one_hot))
num_db2 = numerical_grad(b2, lambda p: forward_loss(W1, b1, W2, p, x, y_one_hot))
num_dW1 = numerical_grad(W1, lambda p: forward_loss(p.reshape(W1.shape), b1, W2, b2, x, y_one_hot))
num_db1 = numerical_grad(b1, lambda p: forward_loss(W1, p, W2, b2, x, y_one_hot))

print("Gradient check  (relative error should be < 1e-5)")
print(f"  W2:  {rel_error(dW2, num_dW2):.2e}")
print(f"  b2:  {rel_error(db2, num_db2):.2e}")
print(f"  W1:  {rel_error(dW1, num_dW1):.2e}")
print(f"  b1:  {rel_error(db1, num_db1):.2e}")

## 5.2 Connection to SGD

Once we have $\nabla_{W^{(l)}}\mathcal{L}$ and $\nabla_{\mathbf{b}^{(l)}}\mathcal{L}$, a gradient descent step is simply:
$$W^{(l)} \;\longleftarrow\; W^{(l)} - \eta\,\nabla_{W^{(l)}}\mathcal{L}, \qquad \mathbf{b}^{(l)} \;\longleftarrow\; \mathbf{b}^{(l)} - \eta\,\nabla_{\mathbf{b}^{(l)}}\mathcal{L},$$
where $\eta > 0$ is the learning rate. This is exactly the update used in [`optimization_sgd.ipynb`](optimization_sgd.ipynb).

In **mini-batch SGD**, the gradients above are averaged over a batch of $B$ examples, and the update is applied after each batch. Keras handles all of this internally; our NumPy implementation here exposes the machinery underneath.

> **Exercise 5.1.** *(Extending to a batch)*
>
> **(a)** Modify the forward/backward code above to handle a batch of $B$ inputs, stored as a matrix $X \in \mathbb{R}^{B \times n_0}$ (one row per example). The gradients should be averaged over the batch.
>
> *Hint:* The weight gradient formula $\nabla_{W^{(l)}}\mathcal{L} = \delta^{(l)}(\mathbf{a}^{(l-1)})^\top$ for a single example generalises to $\frac{1}{B}\Delta^{(l)\top} A^{(l-1)}$ where the columns of $\Delta^{(l)}$ are the individual $\delta^{(l)}$ vectors.
>
> **(b)** Implement one step of gradient descent using your batch backprop, and check that the loss decreases.

> **Exercise 5.2.** *(Modifying the network)*
>
> The code currently uses ReLU hidden units. Adapt the backward pass to use **sigmoid** instead, and re-run the gradient check.
>
> **(a)** What one line of the backward pass needs to change?
>
> **(b)** Does the gradient check still pass? Compare the magnitudes of $\delta^{(1)}$ between the ReLU and sigmoid cases with the same initialisation. What do you observe?

---
# Summary

| Concept | Key formula |
|---------|-------------|
| **Chain rule (scalar)** | $\dfrac{da}{dx} = \dfrac{da}{dz}\cdot\dfrac{dz}{dx}$ |
| **Backprop rule** | $\dfrac{\partial\mathcal{L}}{\partial v} = \dfrac{\partial\mathcal{L}}{\partial u} \cdot \dfrac{\partial u}{\partial v}$ (upstream × local) |
| **Output error** | $\delta^{(L)} = \hat{\mathbf{y}} - \mathbf{y}$ (softmax + cross-entropy) |
| **Hidden error** | $\delta^{(l)} = \bigl(W^{(l+1)\top}\delta^{(l+1)}\bigr)\odot\sigma'(\mathbf{z}^{(l)})$ |
| **Weight gradient** | $\nabla_{W^{(l)}}\mathcal{L} = \delta^{(l)}\bigl(\mathbf{a}^{(l-1)}\bigr)^{\!\top}$ |
| **Bias gradient** | $\nabla_{\mathbf{b}^{(l)}}\mathcal{L} = \delta^{(l)}$ |
| **Cost of backprop** | $O(P)$ operations — same order as a forward pass |

**Key insight:** backpropagation is not a new algorithm — it is simply the chain rule applied systematically to a computational graph, traversed in reverse topological order. Modern deep-learning frameworks (JAX, PyTorch, TensorFlow) implement this automatically via **automatic differentiation**.